In [1]:
# ============================================================
# CÀI ĐẶT THƯ VIỆN
# ============================================================
!pip install pandas
!pip install py_vncorenlp pandas
%pip install transformers torch sentencepiece


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# IMPORT THƯ VIỆN
# ============================================================

import os
import re
import unicodedata
import random

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Cleaning

In [3]:
# ============================================================
# 1. ĐỌC DỮ LIỆU TRAIN / DEV / TEST
# ============================================================

train_file = "vifactcheck_train.csv"
dev_file   = "vifactcheck_dev.csv"
test_file  = "vifactcheck_test.csv"

train_df = pd.read_csv(train_file)
dev_df   = pd.read_csv(dev_file)
test_df  = pd.read_csv(test_file)

In [4]:
# ============================================================
# 2. KIỂM TRA KÍCH THƯỚC DỮ LIỆU
# ============================================================

print("========== KÍCH THƯỚC DỮ LIỆU ==========")

print("Train:", train_df.shape)
print("Dev  :", dev_df.shape)
print("Test :", test_df.shape)

========== KÍCH THƯỚC DỮ LIỆU ==========
Train: (5062, 10)
Dev  : (723, 10)
Test : (1447, 10)


In [5]:
# ============================================================
# 3. KIỂM TRA TÊN CÁC CỘT
# ============================================================

print("\n========== TÊN CÁC CỘT ==========")

print(train_df.columns.tolist())


========== TÊN CÁC CỘT ==========
['Unnamed: 0', 'index', 'Statement', 'Context', 'annotation_id', 'Topic', 'Author', 'Url', 'labels', 'Evidence']


In [6]:
# ============================================================
# 4. HÀM LÀM SẠCH VĂN BẢN
# ============================================================

def clean_text(text):
    """
    Làm sạch văn bản ở mức nhẹ, phù hợp với PhoBERT.

    Các bước:
    1. Xử lý giá trị thiếu
    2. Chuyển dữ liệu về kiểu string
    3. Chuẩn hóa Unicode bằng NFC
    4. Xóa HTML tags
    5. Xóa URL trong nội dung văn bản
    6. Xóa ký tự điều khiển / zero-width
    7. Chuẩn hóa khoảng trắng
    8. Xóa khoảng trắng đầu và cuối
    """

    # --------------------------------------------------------
    # Bước 1: Xử lý giá trị thiếu
    # --------------------------------------------------------

    if pd.isna(text):
        return ""


    # --------------------------------------------------------
    # Bước 2: Chuyển dữ liệu về dạng chuỗi
    # --------------------------------------------------------

    text = str(text)


    # --------------------------------------------------------
    # Bước 3: Chuẩn hóa Unicode
    # --------------------------------------------------------

    text = unicodedata.normalize("NFC", text)


    # --------------------------------------------------------
    # Bước 4: Xóa HTML tags
    #
    # Ví dụ:
    # <p>Nội dung bài viết</p>
    #
    # Sau khi xử lý:
    # Nội dung bài viết
    # --------------------------------------------------------

    text = re.sub(r"<[^>]+>", " ", text)


    # --------------------------------------------------------
    # Bước 5: Xóa URL
    #
    # Ví dụ:
    # https://example.com/article
    # www.example.com
    #
    # Sau khi xử lý:
    # ""
    # --------------------------------------------------------

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )


    # --------------------------------------------------------
    # Bước 6: Xóa ký tự điều khiển và zero-width
    # --------------------------------------------------------

    text = re.sub(
        r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F"
        r"\u200B-\u200D\uFEFF]",
        "",
        text
    )


    # --------------------------------------------------------
    # Bước 7: Chuẩn hóa khoảng trắng
    #
    # Nhiều khoảng trắng / tab / xuống dòng
    # → một khoảng trắng
    # --------------------------------------------------------

    text = re.sub(r"\s+", " ", text)


    # --------------------------------------------------------
    # Bước 8: Xóa khoảng trắng đầu và cuối
    # --------------------------------------------------------

    text = text.strip()


    return text


In [7]:
# ============================================================
# 5. XÁC ĐỊNH CÁC CỘT VĂN BẢN
# ============================================================

text_columns = [
    "Statement",
    "Context",
    "Evidence"
]

In [8]:
# ============================================================
# 6. LÀM SẠCH TRAIN / DEV / TEST
# ============================================================

datasets = {
    "Train": train_df,
    "Dev": dev_df,
    "Test": test_df
}


for dataset_name, df in datasets.items():

    print(f"\nĐang làm sạch tập {dataset_name}...")

    for column in text_columns:

        if column in df.columns:

            df[column] = df[column].apply(clean_text)



Đang làm sạch tập Train...

Đang làm sạch tập Dev...

Đang làm sạch tập Test...


In [9]:
# ============================================================
# 7. KIỂM TRA DỮ LIỆU SAU KHI LÀM SẠCH
# ============================================================

print("\n========== KIỂM TRA SAU KHI LÀM SẠCH ==========")

for dataset_name, df in datasets.items():

    print(f"\n===== {dataset_name} =====")

    for column in text_columns:

        if column in df.columns:

            print(f"\n--- {column} ---")

            print(
                df[column]
                .head(3)
                .to_string(index=False)
            )




========== KIỂM TRA SAU KHI LÀM SẠCH ==========

===== Train =====

--- Statement ---
Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ, ...
Hành vi của Tô Văn Hải là cho phép người khác đ...
SAWACO thông báo tạm ngưng cung cấp nước để thự...

--- Context ---
(Chinhphu.vn) - Đây là mong muốn, gửi gắm của P...
Ngày 24/3, Cơ quan Cảnh sát điều tra Công an tỉ...
(PLO)- Theo Tổng Công ty Cấp nước Sài Gòn (SAWA...

--- Evidence ---
Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó Th...
Tô Văn Hải đã có hành vi cho phép người khác đổ...
SAWACO thông báo tạm ngưng cung cấp nước để thự...

===== Dev =====

--- Statement ---
Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vua...
Nhiều chi bộ chỉ mua báo đảng mà không quan tâm...
Công ty TNHH Mua bán nợ DSP, địa chỉ 91 Pasteur...

--- Context ---
Saigon Morin, khách sạn 4 sao hàng đầu tại Huế,...
(Chinhphu.vn) - Bí thư Trung ương Đảng, Trưởng ...
(NLĐO)- Sau khi mua khoản nợ từ Công ty Mirae A...

--- Evidence ---
Vua hề Charlie Chaplin (vua hề Sác lô) và 

In [10]:
# ============================================================
# 8. LƯU DỮ LIỆU SAU KHI LÀM SẠCH
# ============================================================

train_output = "vifactcheck_train_phobert_cleaned.csv"
dev_output   = "vifactcheck_dev_phobert_cleaned.csv"
test_output  = "vifactcheck_test_phobert_cleaned.csv"


train_df.to_csv(
    train_output,
    index=False,
    encoding="utf-8-sig"
)

dev_df.to_csv(
    dev_output,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    test_output,
    index=False,
    encoding="utf-8-sig"
)


In [11]:
# ============================================================
# 9. THÔNG BÁO HOÀN THÀNH
# ============================================================

print("\n============================================")
print("ĐÃ HOÀN THÀNH BƯỚC DATA CLEANING")
print("============================================")

print("Train →", train_output)
print("Dev   →", dev_output)
print("Test  →", test_output)


ĐÃ HOÀN THÀNH BƯỚC DATA CLEANING
Train → vifactcheck_train_phobert_cleaned.csv
Dev   → vifactcheck_dev_phobert_cleaned.csv
Test  → vifactcheck_test_phobert_cleaned.csv


# word segmentation

In [12]:
# ============================================================
# 1. XÁC ĐỊNH THƯ MỤC PROJECT
# ============================================================

# Lấy thư mục chứa file Python hiện tại
BASE_DIR = "/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project"

print("Project directory:")
print(BASE_DIR)

Project directory:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project


In [13]:
# ============================================================
# 2. ĐỌC DỮ LIỆU SAU DATA CLEANING
# ============================================================

train_file = os.path.join(
    BASE_DIR,
    "vifactcheck_train_phobert_cleaned.csv"
)

dev_file = os.path.join(
    BASE_DIR,
    "vifactcheck_dev_phobert_cleaned.csv"
)

test_file = os.path.join(
    BASE_DIR,
    "vifactcheck_test_phobert_cleaned.csv"
)


# Kiểm tra file có tồn tại không
for file_path in [train_file, dev_file, test_file]:

    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"\nKhông tìm thấy file:\n{file_path}"
        )


# Đọc dữ liệu
train_df = pd.read_csv(train_file)
dev_df = pd.read_csv(dev_file)
test_df = pd.read_csv(test_file)


print("\nKích thước dữ liệu:")
print("Train:", train_df.shape)
print("Dev  :", dev_df.shape)
print("Test :", test_df.shape)




Kích thước dữ liệu:
Train: (5062, 10)
Dev  : (723, 10)
Test : (1447, 10)


In [14]:
# ============================================================
# 3. XÁC ĐỊNH THƯ MỤC VnCoreNLP
# ============================================================

VNCORENLP_DIR = os.path.join(
    BASE_DIR,
    "vncorenlp"
)

print("\nThư mục VnCoreNLP:")
print(VNCORENLP_DIR)



Thư mục VnCoreNLP:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vncorenlp


In [15]:
# ============================================================
# 4. KIỂM TRA MODEL VnCoreNLP
# ============================================================

# File JAR
jar_file = os.path.join(
    VNCORENLP_DIR,
    "VnCoreNLP-1.2.jar"
)

# Model Word Segmentation
wordsegmenter_dir = os.path.join(
    VNCORENLP_DIR,
    "models",
    "wordsegmenter"
)

rdr_file = os.path.join(
    wordsegmenter_dir,
    "wordsegmenter.rdr"
)

vocab_file = os.path.join(
    wordsegmenter_dir,
    "vi-vocab"
)


# Kiểm tra file JAR
if not os.path.isfile(jar_file):

    raise FileNotFoundError(
        "\nKhông tìm thấy VnCoreNLP-1.2.jar tại:\n"
        + jar_file
        + "\n\n"
        "Hãy download VnCoreNLP model trước."
    )


# Kiểm tra wordsegmenter.rdr
if not os.path.isfile(rdr_file):

    raise FileNotFoundError(
        "\nKhông tìm thấy wordsegmenter.rdr tại:\n"
        + rdr_file
        + "\n\n"
        "Model Word Segmentation chưa đầy đủ."
    )


# Kiểm tra vi-vocab
if not os.path.isfile(vocab_file):

    raise FileNotFoundError(
        "\nKhông tìm thấy vi-vocab tại:\n"
        + vocab_file
        + "\n\n"
        "Model Word Segmentation chưa đầy đủ."
    )


print("\nModel VnCoreNLP đầy đủ.")
print(" - VnCoreNLP-1.2.jar: OK")
print(" - wordsegmenter.rdr: OK")
print(" - vi-vocab: OK")



Model VnCoreNLP đầy đủ.
 - VnCoreNLP-1.2.jar: OK
 - wordsegmenter.rdr: OK
 - vi-vocab: OK


In [16]:
# ============================================================
# 5. KHỞI TẠO VnCoreNLP
# ============================================================
import py_vncorenlp

print("\nĐang khởi tạo VnCoreNLP...")

segmenter = py_vncorenlp.VnCoreNLP(
    annotators=["wseg"],
    save_dir=VNCORENLP_DIR
)

print("Đã khởi tạo VnCoreNLP thành công!")



Đang khởi tạo VnCoreNLP...
2026-09-13 18:08:29 INFO  WordSegmenter:24 - Loading Word Segmentation model
Đã khởi tạo VnCoreNLP thành công!


In [17]:
# ============================================================
# 6. HÀM WORD SEGMENTATION
# ============================================================

def word_segment(text):

    # Xử lý giá trị NaN
    if pd.isna(text):
        return ""

    # Chuyển dữ liệu thành string
    text = str(text).strip()

    # Nếu text rỗng
    if text == "":
        return ""

    # VnCoreNLP word segmentation
    segmented_sentences = segmenter.word_segment(text)

    # Ghép các câu lại
    return " ".join(segmented_sentences)


In [18]:
# ============================================================
# 7. CÁC CỘT CẦN TÁCH TỪ
# ============================================================

TEXT_COLUMNS = [
    "Statement",
    "Context",
    "Evidence"
]

In [19]:
# ============================================================
# 8. KIỂM TRA CÁC CỘT
# ============================================================

for column in TEXT_COLUMNS:

    if column not in train_df.columns:
        raise ValueError(
            f"Train không có cột: {column}"
        )

    if column not in dev_df.columns:
        raise ValueError(
            f"Dev không có cột: {column}"
        )

    if column not in test_df.columns:
        raise ValueError(
            f"Test không có cột: {column}"
        )


print("\nCác cột cần Word Segmentation:")
print(TEXT_COLUMNS)



Các cột cần Word Segmentation:
['Statement', 'Context', 'Evidence']


In [20]:
# ============================================================
# 9. WORD SEGMENTATION
# ============================================================

for column in TEXT_COLUMNS:

    print(
        f"\nĐang Word Segmentation cột: {column}"
    )

    train_df[column] = train_df[column].apply(
        word_segment
    )

    dev_df[column] = dev_df[column].apply(
        word_segment
    )

    test_df[column] = test_df[column].apply(
        word_segment
    )



Đang Word Segmentation cột: Statement

Đang Word Segmentation cột: Context

Đang Word Segmentation cột: Evidence


In [21]:
# ============================================================
# 10. TẠO ĐƯỜNG DẪN FILE OUTPUT
# ============================================================

train_output = os.path.join(
    BASE_DIR,
    "vifactcheck_train_phobert_segmented.csv"
)

dev_output = os.path.join(
    BASE_DIR,
    "vifactcheck_dev_phobert_segmented.csv"
)

test_output = os.path.join(
    BASE_DIR,
    "vifactcheck_test_phobert_segmented.csv"
)

In [22]:
# ============================================================
# 11. LƯU DỮ LIỆU
# ============================================================

train_df.to_csv(
    train_output,
    index=False,
    encoding="utf-8-sig"
)

dev_df.to_csv(
    dev_output,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    test_output,
    index=False,
    encoding="utf-8-sig"
)

In [23]:
# ============================================================
# 12. KIỂM TRA KẾT QUẢ
# ============================================================

print("\n")
print("=" * 60)
print("WORD SEGMENTATION HOÀN TẤT")
print("=" * 60)

print("\nFile output:")

print(
    "Train:",
    train_output
)

print(
    "Dev  :",
    dev_output
)

print(
    "Test :",
    test_output
)



WORD SEGMENTATION HOÀN TẤT

File output:
Train: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_train_phobert_segmented.csv
Dev  : /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_dev_phobert_segmented.csv
Test : /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_test_phobert_segmented.csv


In [24]:
# ============================================================
# 13. XEM KẾT QUẢ MỘT SAMPLE
# ============================================================

print("\n")
print("=" * 60)
print("VÍ DỤ KẾT QUẢ")
print("=" * 60)


print("\n[Statement]")
print(train_df["Statement"].iloc[0])


print("\n[Context]")
print(train_df["Context"].iloc[0])


print("\n[Evidence]")
print(train_df["Evidence"].iloc[0])


print("\nĐã hoàn thành bước Word Segmentation.")



VÍ DỤ KẾT QUẢ

[Statement]
Phó Thủ_tướng Trần_Hồng_Hà thay_mặt Chính_phủ , Thủ_tướng Chính_phủ chúc_mừng Đài_Truyền_hình Việt_Nam , Đài_truyền_hình các tỉnh , thành_phố trên cả nước , các đơn_vị sản_xuất truyền_hình và TP. Hải_Phòng sau 2 năm gián_đoạn do đại_dịch COVID-19 đã tổ_chức rất thành_công sự_kiện quan_trọng này .

[Context]
( Chinhphu.vn ) - Đây là mong_muốn , gửi_gắm của Phó Thủ_tướng Trần_Hồng_Hà đến những người làm truyền_hình tại lễ bế_mạc Liên_hoan Truyền_hình toàn_quốc lần thứ 41 , tối 18/3 , tại TP. Hải_Phòng . Phó Thủ_tướng Trần_Hồng_Hà : Các tác_phẩm truyền_hình đã vun_đắp , làm_giàu cho nền văn_hoá Việt_Nam tiên_tiến , đậm_đà bản_sắc dân_tộc , góp_phần tạo_dựng môi_trường văn_hoá lành_mạnh và xây_dựng con_người Việt_Nam nhân_cách , trách_nhiệm , hội_nhập - Ảnh : VGP / Minh_Khôi_Tham dự lễ bế_mạc còn có Bí_thư Trung_ương Đảng , Trưởng Ban Tuyên_giáo Trung_ương Nguyễn_Trọng_Nghĩa , lãnh_đạo các bộ , ngành Trung_ương , địa_phương , đại_diện các đài_truyền_hình , đơn_

## Phobert Tokenization

In [25]:
# ============================================================
# 2. CẤU HÌNH PHOBERT
# ============================================================

MODEL_NAME = "vinai/phobert-base-v2"

MAX_LENGTH = 256

LABEL_COLUMN = "labels"

print("\n" + "=" * 60)
print("PHOBERT CONFIGURATION")
print("=" * 60)

print("Model      :", MODEL_NAME)
print("Max length :", MAX_LENGTH)
print("Label      :", LABEL_COLUMN)


PHOBERT CONFIGURATION
Model      : vinai/phobert-base-v2
Max length : 256
Label      : labels


In [26]:
# ============================================================
# 3. ĐƯỜNG DẪN INPUT / OUTPUT
# ============================================================

train_file = os.path.join(
    BASE_DIR,
    "vifactcheck_train_phobert_segmented.csv"
)

dev_file = os.path.join(
    BASE_DIR,
    "vifactcheck_dev_phobert_segmented.csv"
)

test_file = os.path.join(
    BASE_DIR,
    "vifactcheck_test_phobert_segmented.csv"
)


output_train = os.path.join(
    BASE_DIR,
    "vifactcheck_train_phobert_tokenized.pt"
)

output_dev = os.path.join(
    BASE_DIR,
    "vifactcheck_dev_phobert_tokenized.pt"
)

output_test = os.path.join(
    BASE_DIR,
    "vifactcheck_test_phobert_tokenized.pt"
)

output_label_mapping = os.path.join(
    BASE_DIR,
    "phobert_label_mapping.pt"
)

In [27]:
# ============================================================
# 4. KIỂM TRA FILE INPUT
# ============================================================

print("\n" + "=" * 60)
print("CHECK INPUT FILES")
print("=" * 60)

input_files = {
    "Train": train_file,
    "Dev": dev_file,
    "Test": test_file
}

for name, path in input_files.items():

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"\n{name} file không tồn tại:\n{path}"
        )

    print(f"{name}: OK")
    print(f"  {path}")




CHECK INPUT FILES
Train: OK
  /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_train_phobert_segmented.csv
Dev: OK
  /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_dev_phobert_segmented.csv
Test: OK
  /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_test_phobert_segmented.csv


In [28]:
# ============================================================
# 5. LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

train_df = pd.read_csv(train_file)
dev_df = pd.read_csv(dev_file)
test_df = pd.read_csv(test_file)

print("Train:", train_df.shape)
print("Dev  :", dev_df.shape)
print("Test :", test_df.shape)



LOADING DATA
Train: (5062, 10)
Dev  : (723, 10)
Test : (1447, 10)


In [29]:
# ============================================================
# 6. CHUẨN HÓA TÊN CỘT
# ============================================================

train_df.columns = train_df.columns.str.strip()
dev_df.columns = dev_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()


In [30]:
# ============================================================
# 7. KIỂM TRA CỘT DỮ LIỆU
# ============================================================

REQUIRED_COLUMNS = [
    "Statement",
    "Evidence",
    LABEL_COLUMN
]

print("\n" + "=" * 60)
print("CHECK REQUIRED COLUMNS")
print("=" * 60)

for split_name, df in [
    ("Train", train_df),
    ("Dev", dev_df),
    ("Test", test_df)
]:

    missing_columns = [
        col
        for col in REQUIRED_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"\n{split_name} thiếu các cột: "
            f"{missing_columns}\n"
            f"Các cột hiện có:\n"
            f"{df.columns.tolist()}"
        )

    print(f"{split_name}: OK")




CHECK REQUIRED COLUMNS
Train: OK
Dev: OK
Test: OK


In [31]:
# ============================================================
# 8. KIỂM TRA MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("CHECK MISSING VALUES")
print("=" * 60)

for split_name, df in [
    ("Train", train_df),
    ("Dev", dev_df),
    ("Test", test_df)
]:

    print(f"\n{split_name}:")

    for col in REQUIRED_COLUMNS:

        missing = df[col].isna().sum()

        print(
            f"  {col}: {missing}"
        )



CHECK MISSING VALUES

Train:
  Statement: 0
  Evidence: 0
  labels: 0

Dev:
  Statement: 0
  Evidence: 0
  labels: 0

Test:
  Statement: 0
  Evidence: 0
  labels: 0


In [32]:
# ============================================================
# 9. CHUẨN HÓA TEXT
# ============================================================

def prepare_text(df):

    df = df.copy()

    df["Statement"] = (
        df["Statement"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df["Evidence"] = (
        df["Evidence"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    return df


train_df = prepare_text(train_df)
dev_df = prepare_text(dev_df)
test_df = prepare_text(test_df)



In [33]:
# ============================================================
# 10. KIỂM TRA LABEL
# ============================================================

print("\n" + "=" * 60)
print("CHECK LABELS")
print("=" * 60)

print(
    "Train labels:",
    sorted(train_df[LABEL_COLUMN].unique())
)

print(
    "Dev labels  :",
    sorted(dev_df[LABEL_COLUMN].unique())
)

print(
    "Test labels :",
    sorted(test_df[LABEL_COLUMN].unique())
)



CHECK LABELS
Train labels: [np.int64(0), np.int64(1), np.int64(2)]
Dev labels  : [np.int64(0), np.int64(1), np.int64(2)]
Test labels : [np.int64(0), np.int64(1), np.int64(2)]


In [34]:
# ============================================================
# 11. TẠO LABEL MAPPING
# ============================================================
#
# IMPORTANT:
#
# Chỉ tạo mapping từ TRAIN.
#
# Sau đó dùng cùng mapping cho DEV và TEST.
#
# Ví dụ:
#
# Original label:
#     0 -> 0
#     1 -> 1
#     2 -> 2
#
# ============================================================

print("\n" + "=" * 60)
print("CREATE LABEL MAPPING")
print("=" * 60)


train_unique_labels = sorted(
    train_df[LABEL_COLUMN].unique()
)


label2id = {
    label: idx
    for idx, label in enumerate(train_unique_labels)
}


id2label = {
    idx: label
    for label, idx in label2id.items()
}


print("\nlabel2id:")

for label, idx in label2id.items():

    print(
        f"  {label} -> {idx}"
    )


print("\nid2label:")

for idx, label in id2label.items():

    print(
        f"  {idx} -> {label}"
    )




CREATE LABEL MAPPING

label2id:
  0 -> 0
  1 -> 1
  2 -> 2

id2label:
  0 -> 0
  1 -> 1
  2 -> 2


In [35]:
# ============================================================
# 12. ENCODE LABEL
# ============================================================

def encode_labels(df, split_name):

    original_labels = df[LABEL_COLUMN]

    # --------------------------------------------------------
    # Kiểm tra label chưa xuất hiện trong train
    # --------------------------------------------------------

    unknown_labels = set(
        original_labels.unique()
    ) - set(label2id.keys())

    if unknown_labels:

        raise ValueError(
            f"\n{split_name} có label "
            f"chưa xuất hiện trong TRAIN: "
            f"{unknown_labels}"
        )

    # --------------------------------------------------------
    # Mapping label -> id
    # --------------------------------------------------------

    encoded_labels = original_labels.map(
        label2id
    )

    return encoded_labels.astype(int).tolist()


train_labels = encode_labels(
    train_df,
    "TRAIN"
)

dev_labels = encode_labels(
    dev_df,
    "DEV"
)

test_labels = encode_labels(
    test_df,
    "TEST"
)

In [36]:
# ============================================================
# 13. LOAD PHOBERT TOKENIZER
# ============================================================

print("\n" + "=" * 60)
print("LOADING PHOBERT TOKENIZER")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

print("Tokenizer loaded successfully.")

print("\nSpecial tokens:")
#Đánh dấu bắt đầu câu
print(
    "  BOS:",
    tokenizer.bos_token
)
#Đánh dấu kết thúc câu
print(
    "  EOS:",
    tokenizer.eos_token
)
#Token đệm để các câu có cùng độ dài
print(
    "  PAD:",
    tokenizer.pad_token
)
#Token dùng khi gặp từ/token không nhận diện được
print(
    "  UNK:",
    tokenizer.unk_token
)



LOADING PHOBERT TOKENIZER


Tokenizer loaded successfully.

Special tokens:
  BOS: <s>
  EOS: </s>
  PAD: <pad>
  UNK: <unk>


In [37]:
# ============================================================
# 14. TOKENIZATION FUNCTION
# ============================================================

def tokenize_dataset(
    df,
    labels,
    split_name
):

    print("\n" + "-" * 60)
    print(f"TOKENIZING {split_name}")
    print("-" * 60)

    # --------------------------------------------------------
    # Statement
    # --------------------------------------------------------

    statements = (
        df["Statement"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    # --------------------------------------------------------
    # Evidence
    # --------------------------------------------------------

    evidences = (
        df["Evidence"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    # --------------------------------------------------------
    # PhoBERT Tokenization
    #
    # Input:
    #
    #     Statement
    #     +
    #     Evidence
    #
    # --------------------------------------------------------

    encoded = tokenizer(
        statements,
        evidences,

        padding="max_length",

        truncation=True,

        max_length=MAX_LENGTH,

        return_attention_mask=True,

        return_tensors="pt"
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    labels_tensor = torch.tensor(
        labels,
        dtype=torch.long
    )

    # --------------------------------------------------------
    # Kiểm tra kích thước
    # --------------------------------------------------------

    print(
        "Number of samples :",
        len(statements)
    )

    print(
        "Input IDs shape    :",
        encoded["input_ids"].shape
    )

    print(
        "Attention mask     :",
        encoded["attention_mask"].shape
    )

    print(
        "Labels shape       :",
        labels_tensor.shape
    )

    # --------------------------------------------------------
    # Dataset dictionary
    # --------------------------------------------------------

    tokenized_data = {

        "input_ids":
            encoded["input_ids"],

        "attention_mask":
            encoded["attention_mask"],

        "labels":
            labels_tensor
    }

    return tokenized_data


In [38]:
# ============================================================
# 15. TOKENIZE TRAIN / DEV / TEST
# ============================================================

train_tokenized = tokenize_dataset(
    train_df,
    train_labels,
    "TRAIN"
)

dev_tokenized = tokenize_dataset(
    dev_df,
    dev_labels,
    "DEV"
)

test_tokenized = tokenize_dataset(
    test_df,
    test_labels,
    "TEST"
)



------------------------------------------------------------
TOKENIZING TRAIN
------------------------------------------------------------
Number of samples : 5062
Input IDs shape    : torch.Size([5062, 256])
Attention mask     : torch.Size([5062, 256])
Labels shape       : torch.Size([5062])

------------------------------------------------------------
TOKENIZING DEV
------------------------------------------------------------
Number of samples : 723
Input IDs shape    : torch.Size([723, 256])
Attention mask     : torch.Size([723, 256])
Labels shape       : torch.Size([723])

------------------------------------------------------------
TOKENIZING TEST
------------------------------------------------------------


[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Number of samples : 1447
Input IDs shape    : torch.Size([1447, 256])
Attention mask     : torch.Size([1447, 256])
Labels shape       : torch.Size([1447])


In [39]:
# ============================================================
# 16. SAVE TOKENIZED DATA
# ============================================================

print("\n" + "=" * 60)
print("SAVING TOKENIZED DATA")
print("=" * 60)


torch.save(
    train_tokenized,
    output_train
)

torch.save(
    dev_tokenized,
    output_dev
)

torch.save(
    test_tokenized,
    output_test
)


SAVING TOKENIZED DATA


In [40]:
# ============================================================
# 17. SAVE LABEL MAPPING
# ============================================================

torch.save(
    {
        "label2id": label2id,
        "id2label": id2label
    },
    output_label_mapping
)


print("\nSaved:")

print(
    "Train:",
    output_train
)

print(
    "Dev  :",
    output_dev
)

print(
    "Test :",
    output_test
)

print(
    "Label mapping:",
    output_label_mapping
)



Saved:
Train: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_train_phobert_tokenized.pt
Dev  : /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_dev_phobert_tokenized.pt
Test : /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/vifactcheck_test_phobert_tokenized.pt
Label mapping: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/phobert_label_mapping.pt


In [ ]:
# ============================================================
# 18. KIỂM TRA MỘT SAMPLE
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE TOKENIZATION")
print("=" * 60)


sample_idx = 0


sample_statement = (
    train_df.iloc[
        sample_idx
    ]["Statement"]
)


sample_evidence = (
    train_df.iloc[
        sample_idx
    ]["Evidence"]
)


sample_label = (
    train_df.iloc[
        sample_idx
    ][LABEL_COLUMN]
)


print("\nStatement:")
print(sample_statement)


print("\nEvidence:")
print(sample_evidence)


print("\nOriginal Label:")
print(sample_label)


print("\nEncoded Label:")
print(train_tokenized["labels"][sample_idx])


print("\nInput IDs:")

print(
    train_tokenized[
        "input_ids"
    ][sample_idx]
)


print("\nAttention Mask:")

print(
    train_tokenized[
        "attention_mask"
    ][sample_idx]
)


SAMPLE TOKENIZATION

Statement:
Phó Thủ_tướng Trần_Hồng_Hà thay_mặt Chính_phủ , Thủ_tướng Chính_phủ chúc_mừng Đài_Truyền_hình Việt_Nam , Đài_truyền_hình các tỉnh , thành_phố trên cả nước , các đơn_vị sản_xuất truyền_hình và TP. Hải_Phòng sau 2 năm gián_đoạn do đại_dịch COVID-19 đã tổ_chức rất thành_công sự_kiện quan_trọng này .

Evidence:
Thay_mặt Chính_phủ , Thủ_tướng Chính_phủ , Phó Thủ_tướng Trần_Hồng_Hà chúc_mừng Đài_Truyền_hình Việt_Nam , Đài_truyền_hình các tỉnh , thành_phố trên cả nước , các đơn_vị sản_xuất truyền_hình và TP. Hải_Phòng đã tổ_chức rất thành_công sự_kiện quan_trọng này sau 2 năm gián_đoạn do đại_dịch COVID-19 .

Original Label:
0

Encoded Label:
tensor(0)

Input IDs:
tensor([    0,   268,   324, 17225,  7037,   315,     4,   324,   315,  3367,
         9453,    56,     4,  9914,     9,    98,     4,   214,    34,    94,
           58,     4,     9,   304,   256,  1342,     6,   334,  1343,    53,
           76,    29,  7177,    91, 18759,  9089,  6232,  8745,  11

In [42]:
# ============================================================
# 19. DECODE ĐỂ KIỂM TRA
# ============================================================

decoded_text = tokenizer.decode(
    train_tokenized[
        "input_ids"
    ][sample_idx],

    skip_special_tokens=False
)


print("\nDecoded tokens:")

print(decoded_text)


Decoded tokens:
<s> Phó Thủ_tướng Trần_Hồng_Hà thay_mặt Chính_phủ , Thủ_tướng Chính_phủ chúc_mừng Đài_Truyền_hình Việt_Nam , Đài_truyền_hình các tỉnh , thành_phố trên cả nước , các đơn_vị sản_xuất truyền_hình và TP. Hải_Phòng sau 2 năm gián_đoạn do đại_dịch COVID-19 đã tổ_chức rất thành_công sự_kiện quan_trọng này . </s> </s> Thay_mặt Chính_phủ , Thủ_tướng Chính_phủ , Phó Thủ_tướng Trần_Hồng_Hà chúc_mừng Đài_Truyền_hình Việt_Nam , Đài_truyền_hình các tỉnh , thành_phố trên cả nước , các đơn_vị sản_xuất truyền_hình và TP. Hải_Phòng đã tổ_chức rất thành_công sự_kiện quan_trọng này sau 2 năm gián_đoạn do đại_dịch COVID-19 . </s> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

In [43]:
# ============================================================
# 20. KIỂM TRA LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)


print("\nTRAIN:")

print(
    train_df[LABEL_COLUMN]
    .value_counts()
    .sort_index()
)


print("\nDEV:")

print(
    dev_df[LABEL_COLUMN]
    .value_counts()
    .sort_index()
)


print("\nTEST:")

print(
    test_df[LABEL_COLUMN]
    .value_counts()
    .sort_index()
)



LABEL DISTRIBUTION

TRAIN:
labels
0    1751
1    1658
2    1653
Name: count, dtype: int64

DEV:
labels
0    256
1    244
2    223
Name: count, dtype: int64

TEST:
labels
0    508
1    468
2    471
Name: count, dtype: int64


## Phobert Training

Fine-tuning PhoBERT for Vietnamese Fact-Checking

Input:
   - vifactcheck_train_phobert_tokenized.pt
   - vifactcheck_dev_phobert_tokenized.pt
   - vifactcheck_test_phobert_tokenized.pt
   - phobert_label_mapping.pt

 Model:
   vinai/phobert-base-v2

 Task:
   Statement + Evidence -> Label

In [44]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "vinai/phobert-base-v2"

# ------------------------------------------------------------
# Training parameters
# ------------------------------------------------------------

BATCH_SIZE = 16

EPOCHS = 5

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.1

MAX_GRAD_NORM = 1.0

RANDOM_SEED = 42



In [45]:
# ============================================================
# 3. DEVICE
# ============================================================

print("\n" + "=" * 70)
print("DEVICE")
print("=" * 70)

if torch.backends.mps.is_available():

    DEVICE = torch.device("mps")

elif torch.cuda.is_available():

    DEVICE = torch.device("cuda")

else:

    DEVICE = torch.device("cpu")


print("Using device:", DEVICE)



DEVICE
Using device: mps


In [46]:
# ============================================================
# 4. SET RANDOM SEED
# ============================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)

print("Random seed:", RANDOM_SEED)


Random seed: 42


In [47]:
# ============================================================
# 5. INPUT FILES
# ============================================================

train_file = os.path.join(
    BASE_DIR,
    "vifactcheck_train_phobert_tokenized.pt"
)

dev_file = os.path.join(
    BASE_DIR,
    "vifactcheck_dev_phobert_tokenized.pt"
)

test_file = os.path.join(
    BASE_DIR,
    "vifactcheck_test_phobert_tokenized.pt"
)

label_mapping_file = os.path.join(
    BASE_DIR,
    "phobert_label_mapping.pt"
)

In [48]:
# ============================================================
# 6. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "phobert_model"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

BEST_MODEL_DIR = os.path.join(
    OUTPUT_DIR,
    "best_model"
)

In [49]:
# ============================================================
# 7. CHECK INPUT FILES
# ============================================================

print("\n" + "=" * 70)
print("CHECK INPUT FILES")
print("=" * 70)

input_files = {
    "Train": train_file,
    "Dev": dev_file,
    "Test": test_file,
    "Label mapping": label_mapping_file
}

for name, path in input_files.items():

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"\n{name} file không tồn tại:\n{path}"
        )

    print(f"{name}: OK")



CHECK INPUT FILES
Train: OK
Dev: OK
Test: OK
Label mapping: OK


In [50]:
# ============================================================
# 8. LOAD TOKENIZED DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING TOKENIZED DATA")
print("=" * 70)

# PyTorch 2.6+ mặc định weights_only=True.
# Các file này do chính bước 03 của project tạo ra
# bằng torch.save(), nên sử dụng weights_only=False.

train_data = torch.load(
    train_file,
    map_location="cpu",
    weights_only=False
)

dev_data = torch.load(
    dev_file,
    map_location="cpu",
    weights_only=False
)

test_data = torch.load(
    test_file,
    map_location="cpu",
    weights_only=False
)

label_mapping = torch.load(
    label_mapping_file,
    map_location="cpu",
    weights_only=False
)

print("Train data loaded successfully.")
print("Dev data loaded successfully.")
print("Test data loaded successfully.")
print("Label mapping loaded successfully.")



LOADING TOKENIZED DATA
Train data loaded successfully.
Dev data loaded successfully.
Test data loaded successfully.
Label mapping loaded successfully.


In [51]:
# ============================================================
# 9. LOAD LABEL MAPPING
# ============================================================

label2id = label_mapping["label2id"]

id2label = label_mapping["id2label"]


print("\nLabel mapping:")

print("label2id:")

for label, idx in label2id.items():

    print(
        f"  {label} -> {idx}"
    )


print("\nid2label:")

for idx, label in id2label.items():

    print(
        f"  {idx} -> {label}"
    )


NUM_LABELS = len(label2id)

print(
    "\nNumber of labels:",
    NUM_LABELS
)



Label mapping:
label2id:
  0 -> 0
  1 -> 1
  2 -> 2

id2label:
  0 -> 0
  1 -> 1
  2 -> 2

Number of labels: 3


In [52]:
# ============================================================
# 10. CHECK TOKENIZED DATA
# ============================================================

print("\n" + "=" * 70)
print("CHECK TOKENIZED DATA")
print("=" * 70)


def check_tokenized_data(data, name):

    required_keys = [
        "input_ids",
        "attention_mask",
        "labels"
    ]

    for key in required_keys:

        if key not in data:

            raise ValueError(
                f"{name} thiếu key: {key}"
            )

    print(f"\n{name}")

    print(
        "  input_ids      :",
        data["input_ids"].shape
    )

    print(
        "  attention_mask :",
        data["attention_mask"].shape
    )

    print(
        "  labels         :",
        data["labels"].shape
    )

    # --------------------------------------------------------
    # Check number of samples
    # --------------------------------------------------------

    n_input = data["input_ids"].shape[0]

    n_mask = data["attention_mask"].shape[0]

    n_labels = data["labels"].shape[0]

    if not (
        n_input == n_mask == n_labels
    ):

        raise ValueError(
            f"{name}: số lượng input_ids, "
            f"attention_mask và labels không giống nhau."
        )

    print(
        "  Number samples :",
        n_input
    )


check_tokenized_data(
    train_data,
    "TRAIN"
)

check_tokenized_data(
    dev_data,
    "DEV"
)

check_tokenized_data(
    test_data,
    "TEST"
)



CHECK TOKENIZED DATA

TRAIN
  input_ids      : torch.Size([5062, 256])
  attention_mask : torch.Size([5062, 256])
  labels         : torch.Size([5062])
  Number samples : 5062

DEV
  input_ids      : torch.Size([723, 256])
  attention_mask : torch.Size([723, 256])
  labels         : torch.Size([723])
  Number samples : 723

TEST
  input_ids      : torch.Size([1447, 256])
  attention_mask : torch.Size([1447, 256])
  labels         : torch.Size([1447])
  Number samples : 1447


In [53]:
# ============================================================
# 11. PYTORCH DATASET
# ============================================================

class PhoBERTDataset(Dataset):

    def __init__(self, data):

        self.input_ids = data["input_ids"]

        self.attention_mask = data["attention_mask"]

        self.labels = data["labels"]


    def __len__(self):

        return len(self.labels)


    def __getitem__(self, idx):

        return {
            "input_ids":
                self.input_ids[idx],

            "attention_mask":
                self.attention_mask[idx],

            "labels":
                self.labels[idx]
        }


In [54]:
# ============================================================
# 12. CREATE DATASETS
# ============================================================

train_dataset = PhoBERTDataset(
    train_data
)

dev_dataset = PhoBERTDataset(
    dev_data
)

test_dataset = PhoBERTDataset(
    test_data
)


print("\n" + "=" * 70)
print("DATASET SIZE")
print("=" * 70)

print(
    "Train:",
    len(train_dataset)
)

print(
    "Dev  :",
    len(dev_dataset)
)

print(
    "Test :",
    len(test_dataset)
)



DATASET SIZE
Train: 5062
Dev  : 723
Test : 1447


In [55]:
# ============================================================
# 13. CREATE DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False
)

test_loader = DataLoader(
    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False
)


print("\n" + "=" * 70)
print("DATALOADER")
print("=" * 70)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Dev batches:",
    len(dev_loader)
)

print(
    "Test batches:",
    len(test_loader)
)


DATALOADER
Batch size: 16
Train batches: 317
Dev batches: 46
Test batches: 91


In [56]:
# ============================================================
# 14. LOAD PHOBERT MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING PHOBERT MODEL")
print("=" * 70)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,

    num_labels=NUM_LABELS,

    id2label={
        int(k): str(v)
        for k, v in id2label.items()
    },

    label2id={
        str(k): int(v)
        for k, v in label2id.items()
    }
)


LOADING PHOBERT MODEL


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 23601.20it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [57]:
# ============================================================
# 15. MOVE MODEL TO DEVICE
# ============================================================

model.to(DEVICE)

print(
    "Model loaded:",
    MODEL_NAME
)

print(
    "Number of labels:",
    NUM_LABELS
)

print(
    "Device:",
    DEVICE
)

Model loaded: vinai/phobert-base-v2
Number of labels: 3
Device: mps


In [58]:
# ============================================================
# 16. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)

In [59]:
# ============================================================
# 17. LEARNING RATE SCHEDULER
# ============================================================

total_training_steps = (
    len(train_loader) * EPOCHS
)

warmup_steps = int(
    total_training_steps * WARMUP_RATIO
)


scheduler = get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=warmup_steps,

    num_training_steps=total_training_steps
)


print("\n" + "=" * 70)
print("OPTIMIZER / SCHEDULER")
print("=" * 70)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Total training steps:",
    total_training_steps
)

print(
    "Warmup steps:",
    warmup_steps
)


OPTIMIZER / SCHEDULER
Learning rate: 2e-05
Weight decay: 0.01
Total training steps: 1585
Warmup steps: 158


In [60]:
# ============================================================
# 18. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1
    }


In [61]:
# ============================================================
# 19. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device
):

    model.train()

    total_loss = 0.0

    all_predictions = []

    all_labels = []


    for batch_idx, batch in enumerate(loader):

        # ----------------------------------------------------
        # Move batch to device
        # ----------------------------------------------------

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch[
            "labels"
        ].to(device)


        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            labels=labels
        )


        loss = outputs.loss

        logits = outputs.logits


        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            MAX_GRAD_NORM
        )


        # ----------------------------------------------------
        # Update parameters
        # ----------------------------------------------------

        optimizer.step()

        scheduler.step()


        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_loss += loss.item()


        predictions = torch.argmax(
            logits,
            dim=1
        )


        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
        )

        all_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            batch_idx + 1
        ) % 100 == 0:

            print(
                f"  Batch "
                f"{batch_idx + 1}/"
                f"{len(loader)}"
                f" - Loss: "
                f"{loss.item():.4f}"
            )


    # ========================================================
    # Epoch metrics
    # ========================================================

    average_loss = (
        total_loss /
        len(loader)
    )


    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )


    metrics["loss"] = average_loss


    return metrics

In [62]:
# ============================================================
# 20. EVALUATION FUNCTION
# ============================================================

def evaluate(
    model,
    loader,
    device
):

    model.eval()

    total_loss = 0.0

    all_predictions = []

    all_labels = []


    with torch.no_grad():

        for batch in loader:

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            labels = batch[
                "labels"
            ].to(device)


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels
            )


            loss = outputs.loss

            logits = outputs.logits


            total_loss += loss.item()


            # ------------------------------------------------
            # Prediction
            # ------------------------------------------------

            predictions = torch.argmax(
                logits,
                dim=1
            )


            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    # ========================================================
    # Metrics
    # ========================================================

    average_loss = (
        total_loss /
        len(loader)
    )


    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )


    metrics["loss"] = average_loss


    return (
        metrics,
        all_labels,
        all_predictions
    )


# ============================================================
# 21. TRAINING LOOP
# ============================================================

print("\n" + "=" * 70)
print("START TRAINING")
print("=" * 70)

print(
    "Epochs:",
    EPOCHS
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Learning rate:",
    LEARNING_RATE
)


best_dev_f1 = -1.0

best_epoch = 0

training_history = []


for epoch in range(
    1,
    EPOCHS + 1
):

    print("\n")

    print("=" * 70)

    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )

    print("=" * 70)


    # ========================================================
    # TRAIN
    # ========================================================

    print("\n[TRAIN]")

    train_metrics = train_one_epoch(

        model,

        train_loader,

        optimizer,

        scheduler,

        DEVICE
    )


    print(
        f"Train Loss     : "
        f"{train_metrics['loss']:.4f}"
    )

    print(
        f"Train Accuracy : "
        f"{train_metrics['accuracy']:.4f}"
    )

    print(
        f"Train Precision: "
        f"{train_metrics['precision']:.4f}"
    )

    print(
        f"Train Recall   : "
        f"{train_metrics['recall']:.4f}"
    )

    print(
        f"Train F1       : "
        f"{train_metrics['f1']:.4f}"
    )


    # ========================================================
    # DEV
    # ========================================================

    print("\n[DEV]")

    dev_metrics, _, _ = evaluate(

        model,

        dev_loader,

        DEVICE
    )


    print(
        f"Dev Loss       : "
        f"{dev_metrics['loss']:.4f}"
    )

    print(
        f"Dev Accuracy   : "
        f"{dev_metrics['accuracy']:.4f}"
    )

    print(
        f"Dev Precision  : "
        f"{dev_metrics['precision']:.4f}"
    )

    print(
        f"Dev Recall     : "
        f"{dev_metrics['recall']:.4f}"
    )

    print(
        f"Dev F1         : "
        f"{dev_metrics['f1']:.4f}"
    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    training_history.append({

        "epoch": epoch,

        "train_loss":
            train_metrics["loss"],

        "train_accuracy":
            train_metrics["accuracy"],

        "train_precision":
            train_metrics["precision"],

        "train_recall":
            train_metrics["recall"],

        "train_f1":
            train_metrics["f1"],

        "dev_loss":
            dev_metrics["loss"],

        "dev_accuracy":
            dev_metrics["accuracy"],

        "dev_precision":
            dev_metrics["precision"],

        "dev_recall":
            dev_metrics["recall"],

        "dev_f1":
            dev_metrics["f1"]
    })


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if dev_metrics["f1"] > best_dev_f1:

        best_dev_f1 = (
            dev_metrics["f1"]
        )

        best_epoch = epoch


        print("\n*** NEW BEST MODEL ***")

        print(
            f"Best Dev F1: "
            f"{best_dev_f1:.4f}"
        )


        os.makedirs(
            BEST_MODEL_DIR,
            exist_ok=True
        )


        model.save_pretrained(
            BEST_MODEL_DIR
        )


        # ----------------------------------------------------
        # Save training configuration
        # ----------------------------------------------------

        torch.save(

            {
                "epoch": epoch,

                "best_dev_f1":
                    best_dev_f1,

                "label2id":
                    label2id,

                "id2label":
                    id2label,

                "model_name":
                    MODEL_NAME,

                "max_length":
                    256,

                "batch_size":
                    BATCH_SIZE,

                "learning_rate":
                    LEARNING_RATE,

                "weight_decay":
                    WEIGHT_DECAY
            },

            os.path.join(
                BEST_MODEL_DIR,
                "training_config.pt"
            )
        )



START TRAINING
Epochs: 5
Batch size: 16
Learning rate: 2e-05


EPOCH 1/5

[TRAIN]
  Batch 100/317 - Loss: 1.1101
  Batch 200/317 - Loss: 0.8673
  Batch 300/317 - Loss: 0.6635
Train Loss     : 0.8907
Train Accuracy : 0.5741
Train Precision: 0.5754
Train Recall   : 0.5717
Train F1       : 0.5700

[DEV]
Dev Loss       : 0.5738
Dev Accuracy   : 0.7870
Dev Precision  : 0.7919
Dev Recall     : 0.7902
Dev F1         : 0.7871

*** NEW BEST MODEL ***
Best Dev F1: 0.7871


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.80it/s]




EPOCH 2/5

[TRAIN]
  Batch 100/317 - Loss: 0.5100
  Batch 200/317 - Loss: 0.4500
  Batch 300/317 - Loss: 0.3694
Train Loss     : 0.4528
Train Accuracy : 0.8427
Train Precision: 0.8424
Train Recall   : 0.8424
Train F1       : 0.8423

[DEV]
Dev Loss       : 0.5399
Dev Accuracy   : 0.8036
Dev Precision  : 0.8153
Dev Recall     : 0.8005
Dev F1         : 0.8037

*** NEW BEST MODEL ***
Best Dev F1: 0.8037


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.85it/s]




EPOCH 3/5

[TRAIN]
  Batch 100/317 - Loss: 0.2351
  Batch 200/317 - Loss: 0.3995
  Batch 300/317 - Loss: 0.5275
Train Loss     : 0.2981
Train Accuracy : 0.9060
Train Precision: 0.9062
Train Recall   : 0.9058
Train F1       : 0.9059

[DEV]
Dev Loss       : 0.6072
Dev Accuracy   : 0.8147
Dev Precision  : 0.8248
Dev Recall     : 0.8140
Dev F1         : 0.8159

*** NEW BEST MODEL ***
Best Dev F1: 0.8159


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.73it/s]




EPOCH 4/5

[TRAIN]
  Batch 100/317 - Loss: 0.0387
  Batch 200/317 - Loss: 0.0270
  Batch 300/317 - Loss: 0.0229
Train Loss     : 0.1984
Train Accuracy : 0.9423
Train Precision: 0.9424
Train Recall   : 0.9423
Train F1       : 0.9423

[DEV]
Dev Loss       : 0.6603
Dev Accuracy   : 0.8271
Dev Precision  : 0.8282
Dev Recall     : 0.8285
Dev F1         : 0.8270

*** NEW BEST MODEL ***
Best Dev F1: 0.8270


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.78it/s]




EPOCH 5/5

[TRAIN]
  Batch 100/317 - Loss: 0.0325
  Batch 200/317 - Loss: 0.0179
  Batch 300/317 - Loss: 0.0181
Train Loss     : 0.1429
Train Accuracy : 0.9633
Train Precision: 0.9634
Train Recall   : 0.9632
Train F1       : 0.9632

[DEV]
Dev Loss       : 0.6906
Dev Accuracy   : 0.8326
Dev Precision  : 0.8325
Dev Recall     : 0.8343
Dev F1         : 0.8328

*** NEW BEST MODEL ***
Best Dev F1: 0.8328


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]


In [63]:
# ============================================================
# 22. TRAINING FINISHED
# ============================================================

print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best Dev F1:",
    f"{best_dev_f1:.4f}"
)


TRAINING FINISHED
Best epoch: 5
Best Dev F1: 0.8328


In [64]:
# ============================================================
# 23. LOAD BEST MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING BEST MODEL")
print("=" * 70)


best_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        BEST_MODEL_DIR
    )
)


best_model.to(DEVICE)


LOADING BEST MODEL


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4879.36it/s]


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [65]:
# ============================================================
# 24. FINAL DEV EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DEV EVALUATION")
print("=" * 70)


dev_metrics, dev_true, dev_pred = evaluate(

    best_model,

    dev_loader,

    DEVICE
)


print(
    f"Dev Loss      : "
    f"{dev_metrics['loss']:.4f}"
)

print(
    f"Dev Accuracy  : "
    f"{dev_metrics['accuracy']:.4f}"
)

print(
    f"Dev Precision : "
    f"{dev_metrics['precision']:.4f}"
)

print(
    f"Dev Recall    : "
    f"{dev_metrics['recall']:.4f}"
)

print(
    f"Dev F1        : "
    f"{dev_metrics['f1']:.4f}"
)


FINAL DEV EVALUATION
Dev Loss      : 0.6906
Dev Accuracy  : 0.8326
Dev Precision : 0.8325
Dev Recall    : 0.8343
Dev F1        : 0.8328


In [66]:
# ============================================================
# 25. FINAL TEST EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)


test_metrics, test_true, test_pred = evaluate(

    best_model,

    test_loader,

    DEVICE
)


print(
    f"Test Loss      : "
    f"{test_metrics['loss']:.4f}"
)

print(
    f"Test Accuracy  : "
    f"{test_metrics['accuracy']:.4f}"
)

print(
    f"Test Precision : "
    f"{test_metrics['precision']:.4f}"
)

print(
    f"Test Recall    : "
    f"{test_metrics['recall']:.4f}"
)

print(
    f"Test F1        : "
    f"{test_metrics['f1']:.4f}"
)


FINAL TEST EVALUATION
Test Loss      : 0.6457
Test Accuracy  : 0.8445
Test Precision : 0.8442
Test Recall    : 0.8450
Test F1        : 0.8444


In [67]:
# ============================================================
# 26. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("TEST CLASSIFICATION REPORT")
print("=" * 70)


target_names = [
    str(id2label[i])
    for i in range(NUM_LABELS)
]


print(
    classification_report(

        test_true,

        test_pred,

        labels=list(
            range(NUM_LABELS)
        ),

        target_names=target_names,

        digits=4,

        zero_division=0
    )
)


TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.8604    0.8248    0.8422       508
           1     0.8063    0.8184    0.8123       468
           2     0.8660    0.8917    0.8787       471

    accuracy                         0.8445      1447
   macro avg     0.8442    0.8450    0.8444      1447
weighted avg     0.8447    0.8445    0.8444      1447



In [68]:
# ============================================================
# 27. CONFUSION MATRIX
# ============================================================

print("\n" + "=" * 70)
print("TEST CONFUSION MATRIX")
print("=" * 70)


cm = confusion_matrix(

    test_true,

    test_pred,

    labels=list(
        range(NUM_LABELS)
    )
)


print(cm)


TEST CONFUSION MATRIX
[[419  61  28]
 [ 48 383  37]
 [ 20  31 420]]


In [69]:
# ============================================================
# 28. SAVE TRAINING HISTORY
# ============================================================

history_file = os.path.join(

    OUTPUT_DIR,

    "training_history.csv"
)


import pandas as pd


history_df = pd.DataFrame(
    training_history
)


history_df.to_csv(
    history_file,
    index=False
)


print("\nTraining history saved:")

print(history_file)


Training history saved:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/phobert_model/training_history.csv


In [70]:
# ============================================================
# 29. SAVE TEST PREDICTIONS
# ============================================================

predictions_file = os.path.join(

    OUTPUT_DIR,

    "test_predictions.csv"
)


prediction_df = pd.DataFrame({

    "true_label":
        test_true,

    "predicted_label":
        test_pred

})


prediction_df[
    "true_label_name"
] = prediction_df[
    "true_label"
].map(
    id2label
)


prediction_df[
    "predicted_label_name"
] = prediction_df[
    "predicted_label"
].map(
    id2label
)


prediction_df.to_csv(

    predictions_file,

    index=False
)


print(
    "Test predictions saved:"
)

print(
    predictions_file
)


Test predictions saved:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/phobert_model/test_predictions.csv


In [71]:
# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PHOBERT TRAINING COMPLETED!")
print("=" * 70)


print("\nModel:")
print(MODEL_NAME)


print("\nBest epoch:")
print(best_epoch)


print("\nBest Dev F1:")
print(
    f"{best_dev_f1:.4f}"
)


print("\nFinal Test Results:")

print(
    f"  Accuracy  : "
    f"{test_metrics['accuracy']:.4f}"
)

print(
    f"  Precision : "
    f"{test_metrics['precision']:.4f}"
)

print(
    f"  Recall    : "
    f"{test_metrics['recall']:.4f}"
)

print(
    f"  F1        : "
    f"{test_metrics['f1']:.4f}"
)


print("\nSaved model:")
print(BEST_MODEL_DIR)


print("\nSaved files:")

print(
    "  Model:",
    BEST_MODEL_DIR
)

print(
    "  History:",
    history_file
)

print(
    "  Predictions:",
    predictions_file
)


print("\n" + "=" * 70)
print("DONE")
print("=" * 70)


PHOBERT TRAINING COMPLETED!

Model:
vinai/phobert-base-v2

Best epoch:
5

Best Dev F1:
0.8328

Final Test Results:
  Accuracy  : 0.8445
  Precision : 0.8442
  Recall    : 0.8450
  F1        : 0.8444

Saved model:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/phobert_model/best_model

Saved files:
  Model: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/phobert_model/best_model
  History: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/phobert_model/training_history.csv
  Predictions: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/Project/phobert_model/test_predictions.csv

DONE
